downloading ucimlrepo<br>
since we will be using a health disease dataset from https://archive.ics.uci.edu/dataset/519/heart+failure+clinical+records

In [11]:
!pip install ucimlrepo

importing the modules.

In [12]:
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo

## importing the dataset from the site.

In [13]:
# fetch dataset
heart_failure = fetch_ucirepo(id=519)

# data (as pandas dataframes)
X = heart_failure.data.features
y = heart_failure.data.targets
df = pd.concat([X, y], axis=1)

# metadata
print(heart_failure.metadata)
# variable information
print(heart_failure.variables)

{'uci_id': 519, 'name': 'Heart Failure Clinical Records', 'repository_url': 'https://archive.ics.uci.edu/dataset/519/heart+failure+clinical+records', 'data_url': 'https://archive.ics.uci.edu/static/public/519/data.csv', 'abstract': 'This dataset contains the medical records of 299 patients who had heart failure, collected during their follow-up period, where each patient profile has 13 clinical features.', 'area': 'Health and Medicine', 'tasks': ['Classification', 'Regression', 'Clustering'], 'characteristics': ['Multivariate'], 'num_instances': 299, 'num_features': 12, 'feature_types': ['Integer', 'Real'], 'demographics': ['Age', 'Sex'], 'target_col': ['death_event'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2020, 'last_updated': 'Mon Feb 26 2024', 'dataset_doi': '10.24432/C5Z89R', 'creators': [], 'intro_paper': {'ID': 286, 'type': 'NATIVE', 'title': 'Machine learning can predict survival of patients with heart failure f

In [14]:
print(df.info())
print(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 299 entries, 0 to 298
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   age                       299 non-null    float64
 1   anaemia                   299 non-null    int64  
 2   creatinine_phosphokinase  299 non-null    int64  
 3   diabetes                  299 non-null    int64  
 4   ejection_fraction         299 non-null    int64  
 5   high_blood_pressure       299 non-null    int64  
 6   platelets                 299 non-null    float64
 7   serum_creatinine          299 non-null    float64
 8   serum_sodium              299 non-null    int64  
 9   sex                       299 non-null    int64  
 10  smoking                   299 non-null    int64  
 11  time                      299 non-null    int64  
 12  death_event               299 non-null    int64  
dtypes: float64(3), int64(10)
memory usage: 30.5 KB
None
             

## we will now clean and process the data.

In [15]:
#checking if any column has mission values
print(df.isnull().sum())

age                         0
anaemia                     0
creatinine_phosphokinase    0
diabetes                    0
ejection_fraction           0
high_blood_pressure         0
platelets                   0
serum_creatinine            0
serum_sodium                0
sex                         0
smoking                     0
time                        0
death_event                 0
dtype: int64


#### no columns have missing values, so no imputation is needed here.

## Aggregation.

In [16]:
aggregation = df.groupby('death_event').agg({
    'age': ['mean', 'min', 'max'],
    'ejection_fraction': ['mean', 'median'],
    'serum_sodium': ['mean', 'min'],
    'platelets': ['mean']
})

print(aggregation)

                   age             ejection_fraction        serum_sodium       \
                  mean   min   max              mean median         mean  min   
death_event                                                                     
0            58.761906  40.0  90.0          40.26601   38.0   137.216749  113   
1            65.215281  42.0  95.0          33.46875   30.0   135.375000  116   

                 platelets  
                      mean  
death_event                 
0            266657.489901  
1            256381.044792  


#Discretization

In [17]:
age_bins = [0, 45, 65, 100]

age_labels = [
    'Young',
    'Middle-aged',
    'Elderly'
]

df['age_category'] = pd.cut(
    df['age'],
    bins=age_bins,
    labels=age_labels
)

print(df[['age', 'age_category']].head())

    age age_category
0  75.0      Elderly
1  55.0  Middle-aged
2  65.0  Middle-aged
3  50.0  Middle-aged
4  65.0  Middle-aged


#Binarization

In [18]:
df['low_ejection_fraction'] = (df['ejection_fraction'] < 40).astype(int)
df['high_serum_creatinine'] = (df['serum_creatinine'] >= 1.5).astype(int)
df['low_platelets'] = (df['platelets'] < 150000).astype(int)
print(df[['ejection_fraction', 'low_ejection_fraction', 'serum_creatinine',
          'high_serum_creatinine', 'platelets', 'low_platelets']].head())

   ejection_fraction  low_ejection_fraction  serum_creatinine  \
0                 20                      1               1.9   
1                 38                      1               1.1   
2                 20                      1               1.3   
3                 20                      1               1.9   
4                 20                      1               2.7   

   high_serum_creatinine  platelets  low_platelets  
0                      1  265000.00              0  
1                      0  263358.03              0  
2                      0  162000.00              0  
3                      1  210000.00              0  
4                      1  327000.00              0  


#Sampling
selecting a subset of the entire dataset.

In [19]:
sample = df.sample(
    n=100,
    random_state=42
)

print(sample)

      age  anaemia  creatinine_phosphokinase  diabetes  ejection_fraction  \
281  70.0        0                       582         0                 40   
265  50.0        1                       298         0                 35   
164  45.0        0                      2442         1                 30   
9    80.0        1                       123         0                 35   
77   42.0        0                       102         1                 40   
..    ...      ...                       ...       ...                ...   
119  86.0        0                       582         0                 38   
268  45.0        0                       582         1                 38   
269  40.0        0                       582         1                 35   
67   72.0        1                       110         0                 25   
101  75.0        0                       582         0                 45   

     high_blood_pressure  platelets  serum_creatinine  serum_sodium  sex  \